In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
#  FloodNet — ConvNeXt-Tiny Semi-Supervised Training
#  Paper : "Flood or Non-Flooded" (Jackson et al., Water 2023, 15, 875)
#
#  ConvNeXt (Liu et al.): pure ConvNet modernized to match ViT design
#  choices — patchify stem, depthwise conv, inverted bottleneck,
#  GELU activation, LayerNorm. Outperforms Swin Transformer on
#  ImageNet while staying fully convolutional.
#
#  λ SCHEDULE (per epoch):
#    Epochs  1–10  : λ = 0.0   (labeled only)
#    Epochs 11–30  : λ += 0.1 every 2 epochs  (0.1 → 1.0)
#    Epochs 31–50  : λ = 1.0
#
#  α SCHEDULE (Algorithm 1):
#    Epochs  1–20  : α = 0.0
#    Epochs 21–40  : α ramps 0.0 → 1.0 linearly
#    Epochs 41–50  : α = 1.0
#
#  50 epochs | Adam lr=0.0001 | batch=16 | 80/20 labeled split
# ╚══════════════════════════════════════════════════════════════════════╝

import os, copy, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import ConvNeXt_Tiny_Weights
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# ── 1. PATHS ───────────────────────────────────────────────────────────
BASE = Path(
    "/kaggle/input/datasets/aletbm/"
    "aerial-imagery-dataset-floodnet-challenge/"
    "FloodNet Challenge - Track 1"
)
FLOODED_DIR    = BASE / "Train/Labeled/Flooded/image"
NONFLOODED_DIR = BASE / "Train/Labeled/Non-Flooded/image"
UNLABELED_DIR  = BASE / "Train/Unlabeled/image"
EXTS = {".jpg",".jpeg",".png",".JPG",".JPEG",".PNG"}

print("=== Path Check ===")
for name, p in [("Flooded", FLOODED_DIR),
                ("Non-Flooded", NONFLOODED_DIR),
                ("Unlabeled", UNLABELED_DIR)]:
    print(f"  {'OK' if Path(p).exists() else 'MISSING':7} {name:<14} → {p}")

# ── 2. λ schedule ─────────────────────────────────────────────────────
def get_lambda(ep):
    """ep is 0-indexed. λ=0 for first 10 epochs, +0.1 every 2 after."""
    if ep < 10:
        return 0.0
    step = (ep - 10) // 2
    return min(round(0.1 * (step + 1), 1), 1.0)

# ── 3. α schedule (Algorithm 1, lines 2-7) ────────────────────────────
E_ia, E_fa = 20, 40
a_i,  a_f  = 0.0, 1.0

def get_alpha(ep):
    if ep < E_ia: return a_i
    if ep < E_fa: return ((a_f-a_i)/(E_fa-E_ia))*(ep-E_ia)+a_i
    return a_f

# Preview schedule
print("\n=== λ & α Schedule Preview ===")
print(f"  {'Epoch':>5} │ {'λ':>5} │ {'α':>5} │ Note")
print(f"  {'─'*45}")
prev_lam = -1
for ep in range(50):
    lam, alp = get_lambda(ep), get_alpha(ep)
    note = ""
    if ep == 0:                              note = "labeled only starts"
    elif ep == 10:                           note = "pseudo-labeling starts"
    elif lam == 0.2 and prev_lam != 0.2:    note = "★ paper best λ"
    elif lam == 1.0 and prev_lam < 1.0:     note = "λ maxed out"
    elif ep == E_ia:                         note = "α ramp starts"
    elif ep == E_fa:                         note = "α maxed out"
    if note:
        print(f"  {ep+1:>5} │ {lam:>5.1f} │ {alp:>5.3f} │ {note}")
    prev_lam = lam

# ── 4. Collect & split labeled data 80/20 ─────────────────────────────
def collect(folder, label):
    return [(str(p), label) for p in Path(folder).iterdir() if p.suffix in EXTS]

flooded_s    = collect(FLOODED_DIR,    1)
nonflooded_s = collect(NONFLOODED_DIR, 0)
random.shuffle(flooded_s)
random.shuffle(nonflooded_s)

def split80(lst):
    cut = int(0.8 * len(lst))
    return lst[:cut], lst[cut:]

f_train,  f_val  = split80(flooded_s)
nf_train, nf_val = split80(nonflooded_s)

TRAIN_SAMPLES   = f_train + nf_train
VAL_SAMPLES     = f_val   + nf_val
UNLABELED_PATHS = [str(p) for p in Path(UNLABELED_DIR).iterdir()
                   if p.suffix in EXTS]

n_f_tr  = len(f_train)
n_nf_tr = len(nf_train)
n_tr    = len(TRAIN_SAMPLES)

print(f"\n=== Dataset Split ===")
print(f"  Train     : {n_tr}  ({n_f_tr} flooded | {n_nf_tr} non-flooded)")
print(f"  Val       : {len(VAL_SAMPLES)}  ({len(f_val)} flooded | {len(nf_val)} non-flooded)")
print(f"  Unlabeled : {len(UNLABELED_PATHS)}")

# ── 5. Hyper-parameters ────────────────────────────────────────────────
IMG_SIZE = 224
BATCH    = 16
LR       = 1e-4
E        = 50
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"\nDevice  : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── 6. Transforms ─────────────────────────────────────────────────────
tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ── 7. Dataset classes ────────────────────────────────────────────────
class LabeledDS(Dataset):
    def __init__(self, samples, tfm=None):
        self.samples, self.tfm = samples, tfm
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        path, lbl = self.samples[i]
        img = Image.open(path).convert("RGB")
        if self.tfm: img = self.tfm(img)
        return img, torch.tensor(lbl, dtype=torch.float32)

class UnlabeledDS(Dataset):
    def __init__(self, paths, tfm=None):
        self.paths, self.tfm = paths, tfm
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.tfm: img = self.tfm(img)
        return img, self.paths[i]

# ── 8. Dataloaders ────────────────────────────────────────────────────
cw      = [n_tr/(2*n_nf_tr), n_tr/(2*n_f_tr)]
sw      = [cw[int(s[1])] for s in TRAIN_SAMPLES]
sampler = WeightedRandomSampler(sw, len(sw), replacement=True)

train_ds  = LabeledDS(TRAIN_SAMPLES,     tfm)
val_ds    = LabeledDS(VAL_SAMPLES,       tfm)
unl_ds    = UnlabeledDS(UNLABELED_PATHS, tfm)

train_ldr = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                       num_workers=2, pin_memory=True)
val_ldr   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                       num_workers=2, pin_memory=True)
unl_ldr   = DataLoader(unl_ds,   batch_size=BATCH, shuffle=False,
                       num_workers=2, pin_memory=True)

# ── 9. ConvNeXt-Tiny model ────────────────────────────────────────────
#
#  ConvNeXt design choices modernizing plain ResNet:
#    • Patchify stem  : 4×4 conv stride 4 (like ViT patch embedding)
#    • Depthwise conv : 7×7 depthwise (larger receptive field)
#    • Inverted bottleneck: narrow → wide → narrow (like MobileNet)
#    • Fewer activation functions: one GELU per block
#    • LayerNorm instead of BatchNorm
#    • Separate downsampling layers between stages
#
#  ConvNeXt-Tiny stage config:
#    Stage 1: 96  channels, 3 blocks
#    Stage 2: 192 channels, 3 blocks
#    Stage 3: 384 channels, 9 blocks
#    Stage 4: 768 channels, 3 blocks
#    Total  : ~28M params
#
print("\n=== Building ConvNeXt-Tiny ===")
model = models.convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)

# Replace classifier head: LayerNorm + Linear(768→1)
# Original: AdaptiveAvgPool → Flatten → LayerNorm(768) → Linear(768,1000)
in_features = model.classifier[2].in_features   # 768
model.classifier[2] = nn.Linear(in_features, 1) # binary output

model = model.to(DEVICE)

total_p   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total params     : {total_p/1e6:.1f}M")
print(f"  Trainable params : {trainable/1e6:.1f}M")
print(f"  Input            : {IMG_SIZE}×{IMG_SIZE}×3")
print(f"  Stem             : 4×4 conv stride 4 (patchify)")
print(f"  Depthwise conv   : 7×7 kernel")
print(f"  Activation       : GELU")
print(f"  Normalization    : LayerNorm")
print(f"  Stages           : 96→192→384→768 channels")
print(f"  Output           : 1 node (BCEWithLogitsLoss)")

optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss()

# ── 10. Evaluation ────────────────────────────────────────────────────
def evaluate():
    model.eval()
    probs_all, preds_all, lbls_all = [], [], []
    with torch.no_grad():
        for imgs, lbls in val_ldr:
            logits = model(imgs.to(DEVICE)).squeeze(1)
            probs  = torch.sigmoid(logits).cpu().numpy()
            preds  = (probs >= 0.5).astype(int)
            probs_all.extend(probs)
            preds_all.extend(preds)
            lbls_all.extend(lbls.numpy().astype(int))
    acc  = accuracy_score(lbls_all, preds_all)
    f1   = f1_score(lbls_all,       preds_all, zero_division=0)
    prec = precision_score(lbls_all, preds_all, zero_division=0)
    rec  = recall_score(lbls_all,   preds_all, zero_division=0)
    try:    roc = roc_auc_score(lbls_all, probs_all)
    except: roc = float("nan")
    return acc, f1, prec, rec, roc

# ── 11. Training loop (Algorithm 1) ───────────────────────────────────
history    = []
best_f1    = -1
best_epoch = -1
best_row   = {}
best_state = None

print("\n" + "═"*96)
print(f"{'Ep':>3} │ {'λ':>5} │ {'α':>5} │ {'#Pseudo':>7} │ {'Loss':>8} │ "
      f"{'Acc':>6} │ {'F1':>6} │ {'Prec':>6} │ {'Rec':>6} │ {'AUC':>6}")
print("═"*96)

for ep in range(E):
    model.train()
    lam      = get_lambda(ep)
    alpha    = get_alpha(ep)
    total_loss, n_batches = 0.0, 0
    n_pseudo = 0

    # ── Phase A : labeled pass ───────────────────────────────
    for imgs, lbls in train_ldr:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs).squeeze(1), lbls)
        loss.backward(); optimizer.step()
        total_loss += loss.item(); n_batches += 1

    # ── Phase B : score unlabeled → pseudo-labels ────────────
    #   Algorithm 1 lines 8-15:
    #   p ≤ 0.5 − λ  →  non-flooded (0)
    #   p ≥ 0.5 + λ  →  flooded     (1)
    #   else          →  ignored (uncertain margin)
    if alpha > 0:
        model.eval()
        p_imgs, p_lbls = [], []
        with torch.no_grad():
            for imgs, _ in unl_ldr:
                imgs  = imgs.to(DEVICE)
                probs = torch.sigmoid(
                            model(imgs).squeeze(1)
                        ).cpu().numpy()
                for i, pv in enumerate(probs):
                    if pv <= 0.5 - lam:
                        p_imgs.append(imgs[i].cpu())
                        p_lbls.append(torch.tensor(0.0))
                    elif pv >= 0.5 + lam:
                        p_imgs.append(imgs[i].cpu())
                        p_lbls.append(torch.tensor(1.0))

        n_pseudo = len(p_imgs)

        # ── Phase C : fine-tune on pseudo-labeled ────────────
        if p_imgs:
            model.train()
            for s in range(0, len(p_imgs), BATCH):
                pi = torch.stack(p_imgs[s:s+BATCH]).to(DEVICE)
                pl = torch.stack(p_lbls[s:s+BATCH]).to(DEVICE)
                optimizer.zero_grad()
                loss_u = alpha * criterion(model(pi).squeeze(1), pl)
                loss_u.backward(); optimizer.step()
                total_loss += loss_u.item(); n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    acc, f1, prec, rec, roc = evaluate()

    marker = " ★" if lam == 0.2 else "  "

    row = dict(epoch=ep+1, lambda_=lam, alpha=round(alpha,4),
               n_pseudo=n_pseudo, loss=round(avg_loss,4),
               accuracy=round(acc,4), f1=round(f1,4),
               precision=round(prec,4), recall=round(rec,4),
               roc_auc=round(roc,4))
    history.append(row)

    print(f"{ep+1:3d} │ {lam:5.1f} │ {alpha:5.3f} │ {n_pseudo:7d} │ "
          f"{avg_loss:8.4f} │ {acc:6.4f} │ {f1:6.4f} │ "
          f"{prec:6.4f} │ {rec:6.4f} │ {roc:6.4f}{marker}")

    if f1 > best_f1:
        best_f1    = f1
        best_epoch = ep + 1
        best_row   = row.copy()
        best_state = copy.deepcopy(model.state_dict())

print("═"*96)

# ── 12. Best results summary ──────────────────────────────────────────
print("\n╔═══════════════════════════════════════════════════╗")
print("║   ConvNeXt-Tiny — BEST RESULTS  (val F1)         ║")
print("╠═══════════════════════════════════════════════════╣")
print(f"║  Epoch        : {best_row['epoch']:<33}║")
print(f"║  λ at epoch   : {best_row['lambda_']:<33}║")
print(f"║  α at epoch   : {best_row['alpha']:<33}║")
print(f"║  # Pseudo-lbl : {best_row['n_pseudo']:<33}║")
print(f"║  Loss         : {best_row['loss']:<33}║")
print(f"║  Accuracy     : {best_row['accuracy']:<33}║")
print(f"║  F1 Score     : {best_row['f1']:<33}║")
print(f"║  Precision    : {best_row['precision']:<33}║")
print(f"║  Recall       : {best_row['recall']:<33}║")
print(f"║  ROC-AUC      : {best_row['roc_auc']:<33}║")
print("╚═══════════════════════════════════════════════════╝")

ep_02 = next((r for r in history if r["lambda_"] == 0.2), None)
if ep_02:
    print(f"\n  Results at λ=0.2 (Epoch {ep_02['epoch']} — paper's claimed best):")
    print(f"    Accuracy  = {ep_02['accuracy']:.4f}")
    print(f"    F1 Score  = {ep_02['f1']:.4f}")
    print(f"    Precision = {ep_02['precision']:.4f}")
    print(f"    Recall    = {ep_02['recall']:.4f}")
    print(f"    ROC-AUC   = {ep_02['roc_auc']:.4f}")

# ── 13. Save ──────────────────────────────────────────────────────────
df = pd.DataFrame(history)
df.to_csv("/kaggle/working/convnext_training_history.csv", index=False)
torch.save(best_state, "/kaggle/working/convnext_best.pth")
print(f"\nSaved → convnext_training_history.csv")
print(f"Saved → convnext_best.pth  (epoch {best_epoch}, λ={best_row['lambda_']})")

# ── 14. Training curves ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(
    f"ConvNeXt-Tiny  |  FloodNet  |  λ ramp  |  Best epoch={best_epoch}",
    fontsize=13, fontweight="bold"
)
pairs = [("loss","Loss","tab:red"),
         ("accuracy","Accuracy","tab:blue"),
         ("f1","F1 Score","tab:green"),
         ("precision","Precision","tab:orange"),
         ("recall","Recall","tab:purple"),
         ("roc_auc","ROC-AUC","tab:brown")]

epochs = df["epoch"].values
for ax, (col, title, color) in zip(axes.flat, pairs):
    ax.plot(epochs, df[col], color=color, linewidth=2)
    bv = df.loc[df["epoch"]==best_epoch, col].values[0]
    ax.axvline(best_epoch, color="black", linestyle="--",
               linewidth=1.2, label=f"Best ep {best_epoch}")
    ax.scatter([best_epoch],[bv], color="black", zorder=5, s=70)
    ax.axvspan(1,  10, alpha=0.06, color="blue",   label="λ=0")
    ax.axvspan(10, 30, alpha=0.06, color="orange",  label="λ ramp")
    ax.axvspan(30, 50, alpha=0.06, color="green",   label="λ=1")
    ax.axvline(13, color="red", linestyle=":",
               linewidth=1.2, alpha=0.8, label="λ=0.2 (ep 13)")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=6)
    ax.grid(True, alpha=0.3)

ax_twin = axes.flat[0].twiny()
lam_vals = [get_lambda(ep) for ep in range(E)]
ax_twin.plot(range(1, E+1), lam_vals, color="gray",
             linestyle="--", linewidth=1.2, alpha=0.6)
ax_twin.set_xlabel("λ value (top)", fontsize=8, color="gray")
ax_twin.tick_params(axis="x", labelcolor="gray", labelsize=7)

plt.tight_layout()
plt.savefig("/kaggle/working/convnext_training_curves.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved → convnext_training_curves.png")

# ── 15. Pseudo-label count plot ───────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.bar(df["epoch"], df["n_pseudo"], color="teal", alpha=0.7)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("# Pseudo-labeled samples")
ax2.set_title("ConvNeXt-Tiny — Pseudo-labeled samples per epoch")
ax_r = ax2.twinx()
ax_r.plot(df["epoch"], df["lambda_"], color="red",
          linewidth=2, label="λ schedule")
ax_r.set_ylabel("λ value", color="red")
ax_r.tick_params(axis="y", labelcolor="red")
ax_r.legend(loc="upper right")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/convnext_pseudo_label_counts.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved → convnext_pseudo_label_counts.png")

print("\n✓ ConvNeXt-Tiny training complete.")